# 1.2 Types of Data

**Goal:** classify what data represent and choose sensible summaries, storage checks, and preprocessing steps.

**Prerequisites:** [Understanding Statistics](1_Understanding-Statistics.ipynb). **Runtime:** Python 3.10+, standard library only. Run cells in order; saved outputs are included. All datasets are synthetic.

## Contents
1. Qualitative/categorical and quantitative/numerical data
2. Discrete and continuous data
3. Structured and unstructured data
4. Cross-sectional and time-series data
5. Data dictionaries and an EDA example
6. Practice and solutions

## 1. Qualitative and Quantitative Data; Categorical and Numerical Data

**Qualitative (categorical)** data describe labels, types, or groups: city, device type, subscription plan, or sentiment label. **Quantitative (numerical)** data describe measured or counted quantities: elapsed time, number of purchases, or account balance.

Categorical data can be **nominal** (no intrinsic order, such as browser name) or **ordinal** (ordered, such as low/medium/high). Module 1.3 explains measurement levels. Numeric-looking values can still be categorical: customer IDs, telephone numbers, and postal codes are identifiers, not quantities to average.

Quantitative values support meaningful arithmetic in context. Adding two customer IDs or saying postal code 200000 is “twice” postal code 100000 has no useful interpretation.

| Field | Semantic type | Reason |
|---|---|---|
| `customer_id = "0007"` | Categorical identifier | Leading zeros and identity matter |
| `plan = "premium"` | Categorical | Membership in a group |
| `rating = "good"` | Ordinal categorical | Ordered labels without guaranteed equal gaps |
| `purchases = 3` | Quantitative | A count |
| `duration_seconds = 12.4` | Quantitative | A measured duration |
| `converted = 1` | Binary categorical indicator | Code for yes/no; its mean can represent a proportion |

**Semantic type is not storage type.** A string may contain a number requiring parsing; an integer may encode a category. A float column may hold counts because missing data forced a representation change. Classify using the data dictionary, collection process, and intended use, not only Python types.

In [1]:
rows = [
    {"id": "001", "plan": "basic", "visits": 2, "seconds": 10.2, "converted": 0},
    {"id": "002", "plan": "pro", "visits": 5, "seconds": 14.8, "converted": 1},
    {"id": "003", "plan": "basic", "visits": 0, "seconds": None, "converted": 0},
    {"id": "004", "plan": "pro", "visits": 3, "seconds": 9.0, "converted": 1},
]
for field, value in rows[0].items():
    print(f"{field}: {value!r}, Python storage type={type(value).__name__}")
print("IDs remain strings:", [r["id"] for r in rows])

id: '001', Python storage type=str
plan: 'basic', Python storage type=str
visits: 2, Python storage type=int
seconds: 10.2, Python storage type=float
converted: 0, Python storage type=int
IDs remain strings: ['001', '002', '003', '004']


## 2. Discrete and Continuous Data

A quantitative variable is **discrete** if its possible values form a finite or countably infinite set. Counts are the most common example: visits are $0,1,2,\ldots$. Discrete does not mean “integer storage”: a permitted set of increments such as 0, 0.5, 1.0, … is also discrete.

A **continuous** variable is modeled as taking values across an interval: mass, duration, distance. A recorded measurement is rounded to finite precision, but the underlying quantity may still be continuous. Reporting height as 172 cm does not make underlying height a count.

**Worked example:** number of support tickets is discrete; time to resolve a ticket is continuous. A time rounded to whole minutes is a discretized observation of the continuous duration.

| Distinction | Useful question | Example |
|---|---|---|
| Discrete quantitative | Which countable values are possible? | 0, 1, 2 purchases |
| Continuous quantitative | Is this an underlying measurement on a continuum? | Purchase amount modeled approximately continuously |
| Categorical | Is this a label rather than an amount? | Purchase status code 0, 1, 2 |

Currency has a discrete smallest denomination in a particular system but is often modeled continuously at larger scales. The relevant model and precision should be documented rather than assumed.

**ML implications:** count and continuous features may need different domain checks or modeling assumptions. Do not silently turn missing counts into zero: “not observed” and “observed none” differ.

In [2]:
valid_counts = [0, 2, 5]
invalid_counts = [-1, 2.5]
def valid_nonnegative_count(value):
    return isinstance(value, int) and not isinstance(value, bool) and value >= 0
print("Count checks:", [(x, valid_nonnegative_count(x)) for x in valid_counts + invalid_counts])
durations = [1.24, 1.25, 1.26]
print("Underlying duration examples:", durations)
print("Rounded readings:", [round(x, 1) for x in durations])

Count checks: [(0, True), (2, True), (5, True), (-1, False), (2.5, False)]
Underlying duration examples: [1.24, 1.25, 1.26]
Rounded readings: [1.2, 1.2, 1.3]


## 3. Structured and Unstructured Data

**Structured data** follow an explicit schema, often rows and columns with defined fields and relationships. A relational orders table with order ID, amount, and timestamp is structured.

**Unstructured data** do not directly fit a fixed tabular field schema in their raw content: free text, photos, audio, and videos. These still have structure such as pixels, grammar, and sound patterns; “unstructured” does not mean random or unusable.

**Semi-structured data** have flexible keys, tags, or nested records, such as JSON events and XML. Different records may omit keys or contain variable-length lists. A table cell containing a free-text review does not make the review's language content a simple numeric variable.

**Worked example:** a support event might have structured metadata (`ticket_id`, `created_at`), nested semi-structured fields (`device`), and unstructured text (`message`). The same dataset can combine all three forms.

In ML, text may become token counts or embeddings; images may become pixel tensors or learned features. These transformations create representations, not guarantees of meaningful or unbiased features. Preserve the relationship between each representation, its source unit, and its label.

In [3]:
import json
event = json.loads('{"ticket_id":"T1","device":{"os":"android"},"tags":["billing"],"message":"Payment failed twice"}')
print("Structured ID:", event["ticket_id"])
print("Nested device field:", event["device"]["os"])
print("Raw text:", event["message"])
print("Toy derived text feature (word count):", len(event["message"].split()))
# Word count is a simple demonstration, not a full language representation.

Structured ID: T1
Nested device field: android
Raw text: Payment failed twice
Toy derived text feature (word count): 3


## 4. Cross-sectional Data and Time-series Data

**Cross-sectional data** record many units at one point or within a defined reference period: one income observation per household in a November survey. Cross-sectional does not necessarily mean every row was collected at the exact same instant.

**Time-series data** record a variable or system repeatedly over ordered times: daily total demand at one warehouse. Ordering, spacing, trend, seasonality, and temporal dependence matter. Adjacent points may be correlated, so treating rows as independent may be inappropriate.

**Panel/longitudinal data** record multiple units repeatedly over time, for example monthly spending for each customer. These combine cross-sectional and temporal structure. Repeated cross-sections instead sample possibly different people in each period; they are not automatically a panel.

**Worked example:** the purchase amounts of 100 customers this month form a cross-section. Daily store-wide sales totals form a time series. Each customer's daily purchases form a panel.

**ML split choices:** use a temporal holdout when predicting future periods; keep records from the same entity together where user-level generalization is intended. Random row splits can leak future information or nearly duplicate entity behavior across train and test. Fit preprocessing on training data only.

In [4]:
from datetime import date
# ISO dates sort chronologically; parse dates explicitly for other formats.
daily = [("2026-01-03", 15), ("2026-01-01", 10), ("2026-01-04", 18), ("2026-01-02", 12)]
daily = sorted((date.fromisoformat(day), value) for day, value in daily)
train, test = daily[:3], daily[3:]
print("Chronological training data:", train)
print("Future holdout:", test)
print("One-day changes:", [daily[i][1]-daily[i-1][1] for i in range(1, len(daily))])
assert max(day for day, _ in train) < min(day for day, _ in test)

Chronological training data: [(datetime.date(2026, 1, 1), 10), (datetime.date(2026, 1, 2), 12), (datetime.date(2026, 1, 3), 15)]
Future holdout: [(datetime.date(2026, 1, 4), 18)]
One-day changes: [2, 3, 3]


## 5. Dataset understanding and exploratory data analysis (EDA)

These classifications are **different axes**, not competing labels. A dataset can be structured and time-series, with both categorical and continuous columns.

A **data dictionary** documents each field's meaning, unit, semantic type, valid domain, missing codes, collection timing, and intended role. Also record the row unit and key: one row per session differs from one row per user.

For the session table above, suppose `id` is a session ID, `visits` is the user's prior visit count, `seconds` is this session's duration, and `converted` indicates a purchase. At session start, final duration and purchase status are not yet available; using final duration to predict purchase at the start can leak information even though the field's data type is valid.

### A small EDA checklist

- Count rows and distinct units, check duplicate keys, and inspect missingness.
- Summarize category frequencies and numeric ranges/distributions.
- Check impossible values, unexpected categories, units, and precision.
- Examine class imbalance and relevant groups.
- Identify time ordering and repeated entities before selecting a split.
- Treat observed patterns as exploratory, rather than proof of causality.

In [5]:
import statistics
from collections import Counter

print("Rows:", len(rows), "Unique session IDs:", len({r["id"] for r in rows}))
print("Plan counts:", dict(Counter(r["plan"] for r in rows)))
for field in ["visits", "seconds", "converted"]:
    observed = [r[field] for r in rows if r[field] is not None]
    missing = len(rows)-len(observed)
    print(field, "missing:", missing, "mean among observed:", statistics.mean(observed))
print("Conversion rate:", statistics.mean(r["converted"] for r in rows))
assert sum(r["seconds"] is None for r in rows) == 1

Rows: 4 Unique session IDs: 4
Plan counts: {'basic': 2, 'pro': 2}
visits missing: 0 mean among observed: 2.5
seconds missing: 1 mean among observed: 11.333333333333334
converted missing: 0 mean among observed: 0.5
Conversion rate: 0.5


## Practice

1. Classify postal code, number of purchases, elapsed time, and satisfaction (low/medium/high).
2. An integer column stores blood pressure rounded to the nearest unit. Does storage alone make the underlying variable discrete?
3. Is a JSON event with nested fields structured, semi-structured, or unstructured? Can it contain unstructured content?
4. Classify: one row per school this year; weekly revenue for one store; weekly revenue for each of 20 stores.
5. Why might mean customer ID be meaningless even when IDs are integers?
6. A feature stores session duration as strings such as `"12.5 seconds"`. What should you check before converting it?
7. What is wrong with imputing every missing purchase count as zero without examining how missingness occurred?
8. When predicting tomorrow's demand, why might a random train/test split of daily records overstate usefulness?

## Solutions

1. Postal code: nominal categorical identifier. Purchases: discrete quantitative. Elapsed time: continuous quantitative. Satisfaction: ordinal categorical.
2. No. The measured quantity is modeled continuously; integer storage reflects rounding and precision.
3. Usually semi-structured; a free-text message within it is unstructured content.
4. Cross-sectional; time series; panel/longitudinal, respectively.
5. IDs represent identity, not amounts or an ordered quantitative scale; their arithmetic has no relevant interpretation.
6. Verify units, decimal conventions, allowed ranges, missing/invalid strings, and whether the quantity was available at prediction time. Parse deliberately and preserve errors for review.
7. It confuses “not measured/unknown” with “known none,” potentially changing both the distribution and the meaning of the feature.
8. It can train on future periods and ignore temporal dependence or drift. A chronological holdout better matches forecasting use, though gaps and repeated entities may also need attention.

**Readiness check:** distinguish semantic type, storage representation, data organization, and time structure. Use all four when creating a data dictionary.

**Next:** [1.3 Levels of Measurement](3_Levels-of-Measurement.ipynb).